In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!ls /content/drive/MyDrive/merged_submissions.jsonl

/content/drive/MyDrive/merged_submissions.jsonl


In [3]:
import torch
print(torch.cuda.is_available())  # Should be True

True


In [4]:
!pip install -U bertopic
!pip install emoji
!pip install langdetect
!pip install spacy

In [5]:
import pandas as pd
from networkx.classes import non_neighbors
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
import logging
from sklearn.decomposition import IncrementalPCA
from sklearn.cluster import MiniBatchKMeans
from bertopic.vectorizers import OnlineCountVectorizer
import re
import emoji
from langdetect import detect
from multiprocessing import Pool, cpu_count
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing
import spacy
import time
from tqdm import tqdm
import os
import psutil
import json
import sys
import datetime
from torch.utils.data import DataLoader
import numpy as np
from hdbscan import HDBSCAN

In [6]:
import bertopic
print(bertopic.__version__)

0.17.3


In [7]:
# Set up logging to print to stdout and flush immediately
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [8]:
# Load spaCy once
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

def clean_text(text):
    if not isinstance(text, str) or len(text) < 5:
        return None

    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r"u/\w+|r/\w+|>\s.*", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower()

    if not is_english(text):
        return None

    return text.strip()

def preprocess_texts(texts):
    logger.info(f"Starting preprocessing of {len(texts)} texts...")

    # Clean texts with progress bar
    cleaned = []
    for text in tqdm(texts, desc="Cleaning"):
        result = clean_text(text)
        if result and len(result.split()) > 3:
            cleaned.append(result)

    logger.info(f"Retained {len(cleaned)} texts after cleaning.")

    # Lemmatize with spaCy using progress-aware batching
    lemmatized = []
    logger.info("Starting lemmatization...")
    for doc in tqdm(nlp.pipe(cleaned, batch_size=1000), total=len(cleaned), desc="Lemmatizing"):
        tokens = [t.lemma_ for t in doc if not t.is_stop and t.is_alpha]
        if tokens:
            lemmatized.append(" ".join(tokens))

    logger.info("Lemmatization complete.")
    return lemmatized

In [9]:
def process_docs(chunk):
    chunk = chunk[['title', 'selftext']].fillna('').astype(str)
    texts = (chunk['title'] + ' ' + chunk['selftext']).tolist()
    return preprocess_texts(texts)

In [10]:
def batched_encode(model, docs, batch_size=64):
    embeddings = []
    dataloader = DataLoader(docs, batch_size=batch_size)
    for batch in tqdm(dataloader, desc="Encoding batches"):
        emb = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
        embeddings.append(emb)  # Use append, not extend

    # Combine into a single 2D NumPy array
    return np.vstack(embeddings)

In [14]:
def run_topic_modeling_hdbscan(
    path,
    batch_size=64,
    min_topic_size=30,
    output_dir="bertopic_hdbscan_output",
    sample_size=None
):
    os.makedirs(output_dir, exist_ok=True)

    # Device setup
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    logger.info(f"Using device: {device}")

    # Load full dataset
    df = pd.read_json(path, lines=True, nrows=sample_size)
    docs = process_docs(df)
    if not docs:
        logger.warning("No documents after preprocessing. Exiting.")
        return None

    # Embedding model
    embedding_model = SentenceTransformer(
        'flax-sentence-embeddings/reddit_single-context_mpnet-base',
        device=device
    )

    embeddings = batched_encode(embedding_model, docs, batch_size=batch_size)
    embeddings = embeddings.astype("float64")  # Fix dtype for sklearn compatibility

    # Dimensionality reduction
    umap_model = IncrementalPCA(n_components=15)

    # HDBSCAN clustering
    cluster_model = HDBSCAN(
        min_cluster_size=min_topic_size,
        metric='euclidean',
        prediction_data=True
    )

    # Vectorizer
    vectorizer_model = OnlineCountVectorizer(
        stop_words="english",
        decay=0.005,
        min_df=1,
        ngram_range=(1, 2)
    )

    # BERTopic model
    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=True,
        verbose=True,
        min_topic_size=min_topic_size,
    )

    logger.info("Fitting BERTopic model with HDBSCAN...")
    start_time = time.time()
    topics, probs = topic_model.fit_transform(docs, embeddings)
    logger.info(f"BERTopic fitting completed in {time.time() - start_time:.2f} seconds.")

    # Save model
    final_model_path = os.path.join(output_dir, "bertopic_final_hdbscan.model")
    topic_model.save(
        final_model_path,
        serialization="safetensors",
        save_ctfidf=False,
        save_embedding_model='flax-sentence-embeddings/reddit_single-context_mpnet-base'
    )
    logger.info(f"Model saved at: {final_model_path}")

    # Save topic info with top 30 words
    try:
        topic_info_df = topic_model.get_topic_info()
        topic_info_df["Top30Words"] = topic_info_df["Topic"].apply(
            lambda topic: ", ".join([word for word, _ in topic_model.get_topic(topic)[:30]])
        )
        topic_info_path = os.path.join(output_dir, "topic_info.csv")
        topic_info_df.to_csv(topic_info_path, index=False)
        logger.info(f"Topic info saved to: {topic_info_path}")
    except Exception as e:
        logger.warning(f"Failed to save topic info CSV: {e}")

    # Save metadata
    metadata = {
        "timestamp": datetime.datetime.now().isoformat(),
        "total_docs": len(docs),
        "params": {
            "embedding_model": 'flax-sentence-embeddings/reddit_single-context_mpnet-base',
            "batch_size": batch_size,
            "min_topic_size": min_topic_size,
            "vectorizer": {
                "decay": 0.005,
                "min_df": 1,
                "ngram_range": (1, 2)
            }
        }
    }
    metadata_path = os.path.join(output_dir, "model_metadata.json")
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)
    logger.info(f"Metadata saved to: {metadata_path}")

    return topic_model

In [ ]:
path = "/content/drive/MyDrive/merged_submissions.jsonl"
topic_model = run_topic_modeling_hdbscan(
    path,
    batch_size=128,
    output_dir="bertopic_sample_200k_output_hdbscan",
    sample_size=200_000
)

INFO:__main__:Using device: cuda
INFO:__main__:Starting preprocessing of 200000 texts...
Cleaning: 100%|██████████| 200000/200000 [14:00<00:00, 237.93it/s]
INFO:__main__:Retained 185114 texts after cleaning.
INFO:__main__:Starting lemmatization...
Lemmatizing: 100%|██████████| 185114/185114 [15:35<00:00, 197.84it/s]
INFO:__main__:Lemmatization complete.
Encoding batches: 100%|██████████| 1447/1447 [06:00<00:00,  4.02it/s]
INFO:__main__:Fitting BERTopic model with HDBSCAN...
2025-07-27 16:29:18,650 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-07-27 16:29:47,818 - BERTopic - Dimensionality - Completed ✓
2025-07-27 16:29:47,842 - BERTopic - Cluster - Start clustering the reduced embeddings


In [13]:
def run_topic_modeling_partial(
    path,
    chunk_size=100_000,
    batch_size=64,
    min_topic_size=10,
    checkpoint_every=5,
    output_dir="bertopic_output",
    sample_size=None
):
    os.makedirs(output_dir, exist_ok=True)

    # Device setup
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    logger.info(f"Using device: {device}")

    # Data loader
    reader = pd.read_json(path, lines=True, chunksize=chunk_size, nrows=sample_size)

    # Embedding model
    embedding_model = SentenceTransformer(
        'flax-sentence-embeddings/reddit_single-context_mpnet-base',
        device=device
    )

    # Dimensionality reduction
    umap_model = IncrementalPCA(n_components=15)

    # Clustering
    cluster_model = MiniBatchKMeans(
        n_clusters=100,
        batch_size=1000,
        random_state=42
    )

    # Vectorizer
    vectorizer_model = OnlineCountVectorizer(
        stop_words="english",
        decay=0.05,
        min_df=1,
        ngram_range=(1, 2),
        max_features=100_000
    )

    representation_model = MaximalMarginalRelevance(diversity=0.9, top_n_words=30)

    # BERTopic
    topic_model = BERTopic(
        embedding_model=None,  # We'll use precomputed embeddings
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        nr_topics=20,
        min_topic_size=min_topic_size,
        calculate_probabilities=True,
        verbose=True,
    )

    all_docs = []
    total_chunks = 0

    logger.info("Starting chunked topic modeling...")

    for i, chunk in enumerate(reader):
        start_time = time.time()
        try:
            logger.info(f"--- Processing chunk {i + 1} ---")

            docs = process_docs(chunk)
            if not docs:
                logger.warning(f"Chunk {i + 1} is empty after preprocessing. Skipping.")
                continue

            embeddings = batched_encode(embedding_model, docs, batch_size=batch_size)
            #Fix dtype mismatch
            embeddings = embeddings.astype("float64")

            logger.info(f"Chunk {i + 1}: {len(docs)} documents encoded.")

            if i == 0:
                topic_model.fit(docs, embeddings=embeddings)
                logger.info("Initial fit completed.")
            else:
                topic_model.partial_fit(docs, embeddings=embeddings)
                logger.info(f"Partial fit completed on chunk {i + 1}.")

            all_docs.extend(docs)
            total_chunks += 1

            elapsed = time.time() - start_time
            mem_used = psutil.virtual_memory().percent
            logger.info(f"Chunk {i + 1} done in {elapsed:.2f}s, Memory usage: {mem_used:.1f}%")

            # Save intermediate results
            if (i + 1) % checkpoint_every == 0:
                # Force initialization of topic_mapper_ before saving
                if topic_model.topic_mapper_ is None and all_docs:
                    _ = topic_model.transform(all_docs[:10])
                checkpoint_path = os.path.join(output_dir, f"bertopic_checkpoint_{i+1}.model")
                topic_model.save(checkpoint_path,
                                serialization="safetensors",
                                save_ctfidf=False,
                                save_embedding_model="flax-sentence-embeddings/reddit_single-context_mpnet-base")
                logger.info(f"Checkpoint saved at: {checkpoint_path}")

        except Exception as e:
            logger.error(f"Error processing chunk {i + 1}: {str(e)}", exc_info=True)

    # Force initialization of topic_mapper_ before saving
    if topic_model.topic_mapper_ is None and all_docs:
        _ = topic_model.transform(all_docs[:10])
    # Final model save
    final_model_path = os.path.join(output_dir, "bertopic_final.model")
    topic_model.save(final_model_path,
                 serialization="safetensors",
                 save_ctfidf=False,
                 save_embedding_model="flax-sentence-embeddings/reddit_single-context_mpnet-base")

    logger.info(f"Final model saved at: {final_model_path}")

    # Save topic summary
    try:
        topic_info_df = topic_model.get_topic_info()
        topic_info_path = os.path.join(output_dir, "topic_info.csv")
        # Add a column with top 30 words per topic
        topic_info_df["Top30Words"] = topic_info_df["Topic"].apply(
            lambda topic: ", ".join([word for word, _ in topic_model.get_topic(topic)])
            if topic != -1 else ""
        )
        topic_info_df.to_csv(topic_info_path, index=False)
        logger.info(f"Topic info saved to: {topic_info_path}")
    except Exception as e:
        logger.warning(f"Failed to save topic info CSV: {e}")

    # Save model metadata
    metadata = {
        "timestamp": datetime.datetime.now().isoformat(),
        "chunks_processed": total_chunks,
        "total_docs": len(all_docs),
        "params": {
            "embedding_model": 'flax-sentence-embeddings/reddit_single-context_mpnet-base',
            "chunk_size": chunk_size,
            "batch_size": batch_size,
            "min_topic_size": min_topic_size,
            "n_components": 15,
            "n_clusters": 100,
            "vectorizer": {
                "decay": 0.005,
                "min_df": 1,
                "ngram_range": (1, 2)
            }
        }
    }

    metadata_path = os.path.join(output_dir, "model_metadata.json")
    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)
    logger.info(f"Model metadata saved to: {metadata_path}")

    return topic_model

In [12]:
path = "/content/drive/MyDrive/merged_submissions.jsonl"
topic_model = run_topic_modeling_partial(
    path,
    batch_size=128,
    output_dir="bertopic_sample_200k_50k_output_hdbscan",
    chunk_size=50000,
    sample_size=200_000
)

INFO:__main__:Using device: cuda
INFO:__main__:Starting chunked topic modeling...
INFO:__main__:--- Processing chunk 1 ---
INFO:__main__:Starting preprocessing of 50000 texts...
Cleaning: 100%|██████████| 50000/50000 [03:12<00:00, 259.75it/s]
INFO:__main__:Retained 47375 texts after cleaning.
INFO:__main__:Starting lemmatization...
Lemmatizing: 100%|██████████| 47375/47375 [04:14<00:00, 186.42it/s]
INFO:__main__:Lemmatization complete.
Encoding batches: 100%|██████████| 371/371 [01:28<00:00,  4.20it/s]
INFO:__main__:Chunk 1: 47372 documents encoded.
2025-07-27 15:02:32,777 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-07-27 15:02:39,083 - BERTopic - Dimensionality - Completed ✓
2025-07-27 15:02:39,101 - BERTopic - Cluster - Start clustering the reduced embeddings
/usr/local/lib/python3.11/dist-packages/hdbscan/prediction.py:663: RuntimeWarning: invalid value encountered in scalar divide
  in_cluster_probs = all_points_prob_in_some_cluster(
2025-07-27

AttributeError: 'NoneType' object has no attribute 'items'

In [14]:
loaded_model = BERTopic.load("bertopic_sample_200k_50k_output_hdbscan/bertopic_final.model")
topic_info = loaded_model.get_topic_info()

In [16]:
topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,0,67227,0_mg_supplement_ve_day,"[mg, supplement, ve, day, vitamin, magnesium, ...",NaN
1,1,18543,1_remove_delete_supplement_supplement remove,"[remove, delete, supplement, supplement remove...",NaN
2,2,21371,2_supplement_creatine_vitamin_day,"[supplement, creatine, vitamin, day, remove, h...",NaN
3,3,7409,3_feel_like_ve_help,"[feel, like, ve, help, supplement, know, day, ...",NaN
4,4,8989,4_remove_anxiety_supplement_delete,"[remove, anxiety, supplement, delete, sleep, h...",NaN
5,5,7991,5_remove_delete_good_test,"[remove, delete, good, test, stack, uk, need, ...",NaN
6,6,6298,6_mg_caffeine_delete_day,"[mg, caffeine, delete, day, nac, magnesium, fe...",NaN
7,7,8793,7_vitamin_supplement_brand_good,"[vitamin, supplement, brand, good, magnesium, ...",NaN
8,8,7063,8_remove_cause_effect_magnesium,"[remove, cause, effect, magnesium, covid, vs, ...",NaN
9,9,4993,9_protein_good_powder_pre,"[protein, good, powder, pre, workout, pre work...",NaN


In [ ]:
for col in df.columns:
    print(col)

In [ ]:
print(df.shape)
print(df.info())
print(df.describe(include='all'))

In [ ]:
Missing values by column

In [ ]:
print(df.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
df['created_utc'] = pd.to_datetime(df['created_utc'], unit='s', utc=True)
df['year'] = df['created_utc'].dt.year
year_counts = df['year'].value_counts().sort_index()
print(year_counts)
year_counts = df.groupby(df['created_utc'].dt.year).size()

import matplotlib.pyplot as plt

year_counts.sort_index().plot(kind='bar')
plt.xlabel('Year')
plt.ylabel('Number of Posts')
plt.title('Posts per Year')
plt.tight_layout()
plt.show()

In [ ]:
Topic modelling

In [ ]:
import logging

# Setup basic logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

In [ ]:
import pandas as pd

sample_df = df.sample(n=500, random_state=42).fillna({"title": "", "selftext": ""})
docs = (sample_df['title'] + " " + sample_df['selftext']).tolist()
logger.info(f"Prepared {len(docs)} documents for topic modeling")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

logger.info("Loading embedding model")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

logger.info("Initializing vectorizer and BERTopic")
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.85,
    min_df=2,
    max_features=3000
)
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    nr_topics="auto",
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True  # Enables BERTopic’s built-in progress logging  [oai_citation:2‡Stack Overflow](https://stackoverflow.com/questions/76856592/jupyter-keeps-crashing-when-using-bertopics-fit-transform?utm_source=chatgpt.com)
)

In [ ]:
logger.info("Starting fit_transform() on sample documents")
topics, probs = topic_model.fit_transform(docs)
logger.info("Completed BERTopic modeling")

logger.info("Retrieving and displaying topic info")
topic_info = topic_model.get_topic_info()
topic_info.head()

In [ ]:
try:
    logger.info("Generating interactive topic visualization")
    fig = topic_model.visualize_topics()
    fig.write_html("sample_topic_vis.html")
    logger.info("Saved visualization to sample_topic_vis.html")
except Exception as e:
    logger.error(f"Visualization failed: {e}")